In [1]:
from IPython.display import display
from torchvision.datasets.folder import find_classes

from vasijas.config import load_config
from vasijas.paths import ProjectPaths
from vasijas.metadata import cargar_metadata
from vasijas.entrenamiento_cnn import (
    ConfiguracionTransformacionesImagen,
    crear_transformaciones_imagen,
    seleccionar_dispositivo_procesamiento,
)
from vasijas.evaluacion import (
    cargar_modelo_resnet18_evaluacion,
    crear_loader_imagenes_sin_clase,
    crear_tabla_predicciones_con_metadata,
    guardar_tabla_predicciones,
    predecir_imagenes_sin_clase,
    seleccionar_metadata_sin_clase,
    validar_predicciones_sin_clase,
)

# Predicciones para perfiles completos sin clase

Se seleccionan todas las filas de `../../data/05.cuadrados/metadata.csv` cuya columna **Clase Ana** sea nula, vacía o contenga solo espacios. Cada imagen se relaciona con su metadata mediante **Identificador**, sin depender del orden de los archivos.

Se utiliza el modelo de perfiles completos. Al no haber etiquetas de referencia, se presentan clases propuestas y confianza softmax, sin calcular aciertos ni accuracy. La confianza es la salida softmax máxima del modelo y no representa una probabilidad calibrada de acierto.

## Configuración

In [2]:
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)
DATA_DIR = PATHS.DATA_CUADRADOS
CLASSES_DIR = PATHS.DATA_CUADRADOS_DIVIDIDO
MODEL_PATH = (
    PATHS.OUTPUT_MODELS / "model_resnet18_completos_bs16_lr1e-3_weighted_unfreeze.pth"
)
CSV_PATH = PATHS.OUTPUT_PREDICTIONS / f"predicciones_sin_clase_{MODEL_PATH.stem}.csv"
BATCH_SIZE = 16
TAMANO_IMAGEN = 224
DEVICE = seleccionar_dispositivo_procesamiento()

print(f"Dispositivo: {DEVICE}")
print(f"Metadata: {DATA_DIR / 'metadata.csv'}")
print(f"Modelo: {MODEL_PATH.name}")
print(f"CSV de salida: {CSV_PATH.name}")

Dispositivo: cuda
Metadata: /home/rafael/Dev/vasijas/data/05.cuadrados/metadata.csv
Modelo: model_resnet18_completos_bs16_lr1e-3_weighted_unfreeze.pth
CSV de salida: predicciones_sin_clase_model_resnet18_completos_bs16_lr1e-3_weighted_unfreeze.csv


## Selección de perfiles y preparación de imágenes

La selección se realiza con la metadata. Se exige un único archivo por identificador seleccionado; si falta una imagen o hay coincidencias ambiguas, la ejecución se detiene.

Las clases de salida se leen en el mismo orden alfabético que `ImageFolder` usó al entrenar sobre el dataset de completos dividido. No se toman las carpetas de `05.cuadrados` como clases del modelo, porque incluyen `sin_clase` y clases ausentes del entrenamiento.

In [3]:
metadata_cuadrados = cargar_metadata(DATA_DIR)
metadata_sin_clase = seleccionar_metadata_sin_clase(metadata_cuadrados)
nombres_clases, _ = find_classes(CLASSES_DIR)

# Resize y normalización ImageNet, sin aumentación de datos.
transformacion_inferencia = crear_transformaciones_imagen(
    ConfiguracionTransformacionesImagen(tamano_imagen=TAMANO_IMAGEN)
)[2]
loader_sin_clase = crear_loader_imagenes_sin_clase(
    metadata=metadata_sin_clase,
    data_dir=DATA_DIR,
    transformacion=transformacion_inferencia,
    batch_size=BATCH_SIZE,
    dispositivo=DEVICE,
)

print(f"Registros de metadata: {len(metadata_cuadrados)}")
print(f"Perfiles sin clase e imágenes a procesar: {len(loader_sin_clase.dataset)}")
print("Clases del modelo:", ", ".join(nombres_clases))

Registros de metadata: 1242
Perfiles sin clase e imágenes a procesar: 161
Clases del modelo: clase01, clase02, clase03, clase05, clase06, clase07, clase08, clase09, clase10, clase11


## Carga del modelo e inferencia

Se reconstruye ResNet18 desde los pesos locales, se comprueba la cantidad de clases y se procesa la selección completa, incluido el último lote. La validación exige una predicción por identificador y resultados dentro de las clases del modelo.

In [4]:
model = cargar_modelo_resnet18_evaluacion(
    ruta_modelo=MODEL_PATH,
    nombres_clases=nombres_clases,
    dispositivo=DEVICE,
)
predicciones_sin_clase = predecir_imagenes_sin_clase(
    modelo=model,
    loader=loader_sin_clase,
    nombres_clases=nombres_clases,
    dispositivo=DEVICE,
)
validar_predicciones_sin_clase(
    predicciones_sin_clase, metadata_sin_clase, nombres_clases
)

print(f"Predicciones verificadas: {len(predicciones_sin_clase)}")

Predicciones verificadas: 161


## Resultados y exportación

La tabla conserva todas las columnas de la metadata original, incluida **Clase Ana** sin completar, y agrega `ruta_relativa`, `clase_predicha` y `confianza` (entre 0 y 1). El CSV contiene la tabla completa y no modifica el dataset fuente. La distribución resume las clases propuestas para su revisión.

In [5]:
tabla = crear_tabla_predicciones_con_metadata(
    predicciones=predicciones_sin_clase,
    metadata_original=metadata_cuadrados,
    columnas_prediccion=("ruta_relativa", "clase_predicha", "confianza"),
)
guardar_tabla_predicciones(tabla, CSV_PATH)

print(f"Perfiles procesados: {len(tabla)} | Columnas: {len(tabla.columns)}")
print(f"Tabla completa guardada en: {CSV_PATH}")
display(tabla)

distribucion_predicciones = (
    tabla["clase_predicha"]
    .value_counts()
    .reindex(nombres_clases, fill_value=0)
    .rename_axis("clase_predicha")
    .reset_index(name="cantidad")
)
display(distribucion_predicciones)

Perfiles procesados: 161 | Columnas: 16
Tabla completa guardada en: /home/rafael/Dev/vasijas/output/predictions/predicciones_sin_clase_model_resnet18_completos_bs16_lr1e-3_weighted_unfreeze.csv


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,ruta_relativa,clase_predicha,confianza
0,al_08c,Si,IV,11,B,7,1,NaN,116,52,248,ibero,Almedinilla,sin_clase/al_08c.png,clase05,0.811810
1,al_09d,No,II,11,B,6,1,NaN,250,221,50,ibero,Almedinilla,sin_clase/al_09d.png,clase10,0.999262
2,al_09g,Si,IV,13,B,4,1,NaN,105,76,185,ibero,Almedinilla,sin_clase/al_09g.png,clase06,0.882819
3,al_09i,No,II,2,B,1,2,NaN,150,153,235,ibero,Almedinilla,sin_clase/al_09i.png,clase10,0.941242
4,al_09j,No,II,2,B,1,2,NaN,100,61,120,ibero,Almedinilla,sin_clase/al_09j.png,clase10,0.974772
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,tu_40_2,Si,IV,13,B,5,1,NaN,0,0,0,ibero,Tútugi,sin_clase/tu_40_2.png,clase01,0.934618
157,tu_87_1,Si,IV,1,C,4,1,NaN,86,72,168,ibero,Tútugi,sin_clase/tu_87_1.png,clase06,0.946039
158,tu_87_6,No,II,11,B,2,2,NaN,144,43,30,ibero,Tútugi,sin_clase/tu_87_6.png,clase10,0.989771
159,tu_87_7,No,II,11,B,2,2,NaN,147,44,37,ibero,Tútugi,sin_clase/tu_87_7.png,clase10,0.967897


,clase_predicha,cantidad
0,clase01,7
1,clase02,7
2,clase03,26
3,clase05,7
4,clase06,4
5,clase07,11
6,clase08,30
7,clase09,3
8,clase10,53
9,clase11,13
